# Week 12: Customer Reactivation Windows — Timing Your Win-Back
## Dormancy Detection & Reactivation Survival Analysis

**Masterclass:** [Week 12 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week12_reactivation_windows.html)

**Dataset:** CDNOW transactions (bundled with `lifetimes`)

**What You'll Build:**
1. Personal inter-purchase time (IPT) calculation
2. Dormancy threshold detection (k x median IPT)
3. Survival analysis of dormant customers (time-to-reactivation)
4. Automated escalation ladder with ROI triggers

In [ ]:
!pip install lifetimes lifelines pandas matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from lifetimes.datasets import load_transaction_data
from lifelines import KaplanMeierFitter, WeibullAFTFitter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Personal Inter-Purchase Time

In [ ]:
txns = load_transaction_data()
txns['date'] = pd.to_datetime(txns['date'])
txns = txns.sort_values(['id', 'date'])

# Compute inter-purchase times per customer
def compute_ipt(group):
    dates = group['date'].values
    if len(dates) < 2:
        return pd.DataFrame()
    gaps = np.diff(dates).astype('timedelta64[D]').astype(int)
    return pd.DataFrame({
        'id': group['id'].iloc[0],
        'ipt_days': gaps,
        'purchase_num': range(2, len(dates) + 1)
    })

ipt_df = txns.groupby('id').apply(compute_ipt).reset_index(drop=True)

# Customer-level summary
customer_ipt = ipt_df.groupby('id')['ipt_days'].agg(['median', 'mean', 'std', 'count']).reset_index()
customer_ipt.columns = ['id', 'median_ipt', 'mean_ipt', 'std_ipt', 'n_gaps']

print(f"Customers with 2+ purchases: {len(customer_ipt):,}")
print(f"\nMedian IPT distribution:")
print(customer_ipt['median_ipt'].describe().round(1))

In [ ]:
# Visualize IPT distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(customer_ipt['median_ipt'].clip(upper=200), bins=50,
        color='#264653', edgecolor='white', alpha=0.85)
ax.axvline(x=customer_ipt['median_ipt'].median(), color='#c45d3e',
           linestyle='--', linewidth=2, label=f'Population median: {customer_ipt["median_ipt"].median():.0f} days')
ax.set_title('Distribution of Personal Median Inter-Purchase Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Median IPT (Days)')
ax.set_ylabel('Customers')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 2: Dormancy Detection

In [ ]:
# Dormancy threshold: customer is dormant when current gap > k * median_ipt
K = 2.0  # dormancy multiplier

last_purchase = txns.groupby('id')['date'].max().rename('last_purchase')
cutoff = txns['date'].max()

dormancy = customer_ipt.merge(last_purchase, on='id')
dormancy['current_gap'] = (cutoff - dormancy['last_purchase']).dt.days
dormancy['threshold'] = K * dormancy['median_ipt']
dormancy['is_dormant'] = dormancy['current_gap'] > dormancy['threshold']
dormancy['days_over_threshold'] = np.maximum(0, dormancy['current_gap'] - dormancy['threshold'])

print(f"=== Dormancy Analysis (k={K}) ===")
print(f"Active: {(~dormancy['is_dormant']).sum():,} ({(~dormancy['is_dormant']).mean():.1%})")
print(f"Dormant: {dormancy['is_dormant'].sum():,} ({dormancy['is_dormant'].mean():.1%})")
print(f"\nDormant customers - days over threshold:")
print(dormancy.loc[dormancy['is_dormant'], 'days_over_threshold'].describe().round(0))

---
## Part 3: Reactivation Survival Analysis

In [ ]:
# For each dormancy episode: did the customer come back?
# We look at customers who BECAME dormant at some point during the observation period
# and model time-from-dormancy-entry to reactivation

# Simulate reactivation events from the data
dormant_customers = dormancy[dormancy['is_dormant']].copy()
dormant_customers['duration'] = dormant_customers['days_over_threshold']
# Assume: if they have purchases after their dormancy threshold, they reactivated
# For this dataset (fixed endpoint), treat all as censored unless they appear again
dormant_customers['reactivated'] = 0  # simplified: all censored at cutoff

# For demonstration, randomly mark some as reactivated based on recency
np.random.seed(42)
p_react = np.exp(-0.01 * dormant_customers['days_over_threshold'])
dormant_customers['reactivated'] = (np.random.rand(len(dormant_customers)) < p_react * 0.3).astype(int)

# KM of dormant customer reactivation
kmf = KaplanMeierFitter()
kmf.fit(dormant_customers['duration'], dormant_customers['reactivated'],
        label='Dormant Customers')

fig, ax = plt.subplots(figsize=(10, 6))
(1 - kmf.survival_function_).plot(ax=ax, color='#2d6a4f', linewidth=2)
ax.set_title('Cumulative Reactivation Probability (Dormant Customers)', fontsize=14, fontweight='bold')
ax.set_xlabel('Days Since Entering Dormancy')
ax.set_ylabel('P(Reactivated)')
ax.axhline(y=0.1, color='#c45d3e', linestyle='--', alpha=0.5, label='10% reactivation')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 4: Escalation Ladder

| Trigger | Action | Cost | Target |
|---------|--------|------|--------|
| Dormancy entry (day 0) | Personalized email | $0.10 | All dormant |
| Day 14 | Targeted offer (10% off) | $2-5 | High CLV dormant |
| Day 30 | Direct outreach / call | $15-30 | Top 10% CLV only |
| Day 60+ | Move to passive pool | $0 | All remaining |

---
## Exercises

### Exercise 1: Sensitivity to k
Try k = 1.5, 2.0, 2.5, 3.0. How does the dormancy rate change? Plot dormancy % vs. k.

### Exercise 2: CLV-Weighted Prioritization
Merge CLV data from Week 4. Among dormant customers, what % of total CLV is at stake? Who are the top 20 dormant customers by CLV?

### Exercise 3: Optimal Intervention Timing
Using the reactivation curve, at what day does the weekly reactivation hazard drop below 1%? This is your "stop investing" point.

---
## Key Takeaways
1. **Personal IPT** is better than population averages for dormancy detection
2. **k x median IPT** gives a personalized dormancy threshold
3. **Reactivation probability decays fast** — early intervention matters
4. **Escalation ladders** match investment to probability of success